Alunos:
*   *Thiago Corrêa Brandão*
*   *Fabiano Amorim*
*   *Breno Alcaraz*
*   *Isabella Cristina*
*   *João Pedro Menezes*

In [ ]:
# Instalar pacotes necessários
!pip install pycaret
!pip install ydata-profiling
!pip install optuna
!pip install optuna-integration[sklearn]
!pip install optuna


In [ ]:
# Importando bibliotecas necessárias
import pandas as pd
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from ydata_profiling import ProfileReport
from pycaret.classification import *

In [ ]:
# Importando a base
url = "https://github.com/alvaroriz/datascience_datasets/raw/refs/heads/main/iccmh2020rccsm_p.csv"
df = pd.read_csv(url)

In [ ]:
# Verificando o tamanho da base
df.shape

In [ ]:
# Separando uma parte da base para validação ao final
base_trabalho = df.sample(frac=0.9, random_state=42)
base_validacao = df.drop(base_trabalho.index)

print('Base de trabalho',base_trabalho.shape)
print('Base de validação',base_validacao.shape)

In [ ]:
relatorio_inicial = ProfileReport(base_trabalho,title="Relátorio inical",explorative=True)

In [ ]:
# Gerando relátorio inicial da base
relatorio_inicial.to_notebook_iframe()
relatorio_inicial.to_file("relatorio_inicial.html")

**VERDATE** -> Excluir

Esta variável se trata da data da coleta dos registros, que é única pois todos os registros foram coletados no mesmo dia, ou seja, é constante, portanto não agrega nenhum valor em nossa análise.

**PUMFID** -> Excluir

Esta variável se trata dos identificadores únicos de cada registros(ID), portanto não agrega nenhum valor em nossa análise.

**PUMFFACT** -> Excluir

Esta variável foi criada para corrigir viéses da pesquisa(que teve uma amostragem não estatística pois as respostas da pesquisa foram de forma voluntária), portanto não agregam valor, visto que não são microdados dos entrevistados

In [ ]:
base_trabalho = base_trabalho.drop(columns=['VERDATE','PUMFID','PUMFFACT'])
base_trabalho.shape

**Variável alvo**

Defineremos nossa variável alvo como a 'ANXDVGAC', que classifica se a pessoa chegou no ponto de corte da ansiedade generalizada ou não, também irems excluir os NAs da variável alvo, pois não podemos construir um modelo com uma classe rara e que representa os NAs da base.

In [ ]:
base_trabalho = base_trabalho.drop(columns=[
  # Perguntas que definem diretamente o alvo ANXDVGAC. Não podem ser usadas pois geram data leakage.
  'MH_15A',
  'MH_15B',
  'MH_15C',
  'MH_15D',
  'MH_15E',
  'MH_15F',
  'MH_15G',
  'MH_05',

  # Variáveis derivadas/duplicadas do alvo (outra forma de passar a mesma informação)
  'ANXDVGAD',

  'ANXDVSEV',

  # Derivadas de outras variáveis, identificadores ou metadados (não ajudam a prever)
  'MHDVMHI',   # resumo de outra pergunta, redundante (melhor usar a original)

  # Redundância demográfica (já existe outra coluna mais detalhada)
  'AGEGR_10',  # idade em blocos de 10 anos. Redundante, já existe idade mais detalhada
  'PRURURB',   # rural/urbano simples. Redundante, já existe classificação mais completa (PCSIZMIZ)

  # Variáveis muito correlacionadas (dizem quase a mesma coisa de outra que ficou)
  'BH_55D',    # preocupação com saúde da população canadense. Muito parecida com saúde mundial
  'BH_55I',    # capacidade de cooperar durante a crise. Muito parecida com após a crise

  # Variáveis muito desbalanceadas (quase todos respondem igual, pouco útil ao modelo)
  'PBH_55L',   # preocupação com violência doméstica. Quase todo mundo disse “nada preocupado”
  'BH_60B',    # usou delivery de mercado/farmácia. Maioria absoluta disse “nunca”
  'BH_60A',    # frequência de compras no mercado/farmácia. Respostas muito concentradas em 1 faixa
  'BH_60C',    # usou delivery de comida pronta. Maioria disse “nunca”
  'PIIDFLAG',  # identidade indígena. Minoria muito pequena na amostra (desbalanceada)
  'PIMMST',    # status de imigração. Maioria esmagadora não é imigrante
  'PVISMIN',   # minoria visível. Maioria esmagadora não é minoria

])
base_trabalho.shape

In [ ]:
# Dropando valores duplicados
base_trabalho = base_trabalho.drop_duplicates()
# Retirando os registros onde ANXDVGAC são iguais a 9(representação de NA na base)
base_trabalho = base_trabalho[base_trabalho['ANXDVGAC'] != 9]
display(base_trabalho.shape)

In [ ]:
relatorio_posdrops = ProfileReport(base_trabalho,title="Relátorio Pós Mudanças",explorative=True)
relatorio_posdrops.to_notebook_iframe()
relatorio_posdrops.to_file("relatorio_posdrops.html")

In [ ]:
# verificando o balanceamento para decidir se é necessário balancear
base_trabalho['ANXDVGAC'].value_counts()

# plotando a quantidade de cada classe da variável target anxdvgac com percentuais
ax = base_trabalho['ANXDVGAC'].value_counts().plot(kind='bar')

# Adicionando os valores e percentuais nas barras
total = len(base_trabalho['ANXDVGAC'])
for i, v in enumerate(base_trabalho['ANXDVGAC'].value_counts()):
    percentual = (v / total) * 100
    ax.text(i, v + (v * 0.01), f'{v}\n({percentual:.1f}%)',
            ha='center', va='bottom', fontsize=10)

# com base nos resultados, é necessário balancear, para não ter risco do modelo viciar em classificar na categoria predominante

In [ ]:
for i in base_trabalho.columns:
  print(i)
  print(base_trabalho[i].value_counts())

In [ ]:
# excluir todos os registros que tenham 9(representativo de NA para essa base) em alguma variável
for i in base_trabalho.columns:
  base_trabalho = base_trabalho[base_trabalho[i] != 9]
base_trabalho.shape

In [ ]:
from sklearn.model_selection import train_test_split
X = base_trabalho.drop('ANXDVGAC', axis=1)
y = base_trabalho['ANXDVGAC']

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
smote = SMOTE(random_state=42)
X_treino_balanced, y_treino_balanced = smote.fit_resample(X_treino, y_treino)
base_treino_final = pd.concat([
    pd.DataFrame(X_treino_balanced),
    pd.DataFrame(y_treino_balanced)
], axis=1)
base_treino_final['ANXDVGAC'].value_counts()

In [ ]:
# verificando o balanceamento para decidir se é necessário balancear
base_treino_final['ANXDVGAC'].value_counts()

# plotando a quantidade de cada classe da variável target anxdvgac com percentuais
ax = base_treino_final['ANXDVGAC'].value_counts().plot(kind='bar')

# Adicionando os valores e percentuais nas barras
total = len(base_treino_final['ANXDVGAC'])
for i, v in enumerate(base_treino_final['ANXDVGAC'].value_counts()):
    percentual = (v / total) * 100
    ax.text(i, v + (v * 0.01), f'{v}\n({percentual:.1f}%)',
            ha='center', va='bottom', fontsize=10)

# com base nos resultados, é necessário balancear, para não ter risco do modelo viciar em classificar na categoria predominante

In [ ]:
# Sabemos por conta do Pycaret que o melhor modelo é o Extra Tree Classifier
# Divindo a base de trabalho em treino e teste
X = base_treino_final.drop('ANXDVGAC', axis=1)
y = base_treino_final['ANXDVGAC']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
et_classifier = ExtraTreesClassifier(
    n_estimators=100,        # Número de árvores
    random_state=42,         # Para reproducibilidade
    max_depth=None,          # Profundidade máxima (None = ilimitado)
    min_samples_split=2,     # Mínimo de amostras para dividir
    min_samples_leaf=1,      # Mínimo de amostras nas folhas
    bootstrap=True,          # Amostragem com reposição
    n_jobs=-1               # Usar todos os processadores
)

In [ ]:
# Treinar o modelo
et_classifier.fit(X_train, y_train)

In [ ]:
# 5. Fazer previsões
y_pred = et_classifier.predict(X_test)
y_pred_proba = et_classifier.predict_proba(X_test)

In [ ]:
# 6. Avaliar o modelo
print("Acurácia:", accuracy_score(y_test, y_pred))
print("\nRelatório de Classificação:")
print(classification_report(y_test, y_pred))

In [ ]:
# 7. Matriz de confusão
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Matriz de Confusão - Extra Trees Classifier')
plt.ylabel('Valor Real')
plt.xlabel('Previsão')
plt.show()